# 03 — Streaming Demo & Performance Analysis (AA4.3)

This notebook demonstrates the Spark Structured Streaming pipeline and presents
streaming-specific performance metrics as required by **AA4.3**.

## Architecture recap
```
charts.csv  →  Kafka topic 'charts-feed'
                    ↓
         Spark Structured Streaming
          (readStream · foreachBatch)
                    ↓
       Bronze / Silver / Gold  (Delta)
                    ↓
        Dash dashboard  (dcc.Interval 30 s)
```

### How to run the streaming pipeline before this notebook
```bash
make streaming-up          # starts Kafka + producer + consumer
make streaming-logs        # tail logs to watch batches arrive
# wait 3–5 minutes for several batches to accumulate
# then open this notebook
```

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Locate streaming_metrics.jsonl
REPORTS_DIR = Path(os.getenv("REPORTS_DIR", "/home/jovyan/reports"))
METRICS_FILE = REPORTS_DIR / "streaming_metrics.jsonl"

print(f"Looking for metrics at: {METRICS_FILE}")
print(f"File exists: {METRICS_FILE.exists()}")

In [ ]:
# Load all batch metric records
records = []
with METRICS_FILE.open() as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

metrics = pd.DataFrame(records)
metrics["ts"] = pd.to_datetime(metrics["ts"])
print(f"Loaded {len(metrics)} batch records")
metrics.head()

## Summary statistics

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total batches processed",
        "Total rows ingested",
        "Avg rows per batch",
        "Avg processing latency (ms)",
        "Max processing latency (ms)",
        "Avg throughput (rows/s)",
        "Peak throughput (rows/s)",
    ],
    "Value": [
        len(metrics),
        int(metrics["rows"].sum()),
        round(metrics["rows"].mean(), 1),
        round(metrics["processing_ms"].mean(), 1),
        round(metrics["processing_ms"].max(), 1),
        round(metrics["throughput_rows_per_sec"].mean(), 1),
        round(metrics["throughput_rows_per_sec"].max(), 1),
    ]
})
summary

## Throughput over time

Throughput = rows processed per second per micro-batch. Higher is better.
Early batches may show lower throughput due to JVM warm-up and Delta table initialisation.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=metrics["batch_id"],
    y=metrics["throughput_rows_per_sec"],
    mode="lines+markers",
    name="Throughput (rows/s)",
    line=dict(color="#6E5BFF", width=2),
    marker=dict(size=6),
    fill="tozeroy",
    fillcolor="rgba(110,91,255,0.10)",
))
# Average line
avg_tp = metrics["throughput_rows_per_sec"].mean()
fig.add_hline(y=avg_tp, line_dash="dash", line_color="#FF6B6B",
              annotation_text=f"avg {avg_tp:.0f} rows/s")

fig.update_layout(
    title="Streaming Throughput per Micro-Batch",
    xaxis_title="Batch ID",
    yaxis_title="Throughput (rows / second)",
    height=400,
)
fig.show()

## Processing latency per micro-batch

Latency = total wall-clock time for one micro-batch (Bronze write + Silver MERGE +
Gold recompute). This is the end-to-end delay from Kafka message arrival to Delta
persistence. Spark's 10-second trigger means the *maximum detectable latency* before
the dashboard refreshes is ~40 s (10 s trigger + 30 s Dash interval).

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=metrics["batch_id"],
    y=metrics["processing_ms"],
    name="Latency (ms)",
    marker=dict(color="#FF6B6B", opacity=0.8),
))
avg_lat = metrics["processing_ms"].mean()
fig2.add_hline(y=avg_lat, line_dash="dash", line_color="#6E5BFF",
               annotation_text=f"avg {avg_lat:.0f} ms")

fig2.update_layout(
    title="Processing Latency per Micro-Batch (Bronze write + Silver MERGE + Gold recompute)",
    xaxis_title="Batch ID",
    yaxis_title="Latency (ms)",
    height=400,
)
fig2.show()

## Cumulative rows ingested

Shows the Bronze streaming table growing over time — evidence that data is being
continuously persisted in Delta format.

In [ ]:
fig3 = go.Figure(go.Scatter(
    x=metrics["batch_id"],
    y=metrics["bronze_total_rows"],
    mode="lines+markers",
    name="Bronze rows (cumulative)",
    line=dict(color="#4CAF50", width=2.5),
    fill="tozeroy",
    fillcolor="rgba(76,175,80,0.10)",
))
fig3.update_layout(
    title="Cumulative Rows in bronze/charts_streaming (Delta)",
    xaxis_title="Batch ID",
    yaxis_title="Row count",
    height=360,
)
fig3.show()

## Verify streaming Gold table exists

The dashboard reads `gold/top_artists_streaming` to render the live feed section.

In [ ]:
import sys
sys.path.insert(0, "/src")
from bigdata_music import config

from pyspark.sql import SparkSession
from bigdata_music.spark_session import get_spark

spark = get_spark()

for label, path in [
    ("bronze/charts_streaming",    config.BRONZE_CHARTS_STREAMING),
    ("silver/charts_streaming",    config.SILVER_CHARTS_STREAMING),
    ("gold/top_artists_streaming", config.GOLD_TOP_ARTISTS_STREAMING),
]:
    from pathlib import Path
    exists = Path(path).exists()
    if exists:
        n = spark.read.format("delta").load(path).count()
        print(f"✓ {label}: {n:,} rows")
    else:
        print(f"✗ {label}: not found (run 'make streaming-up' first)")

## Analysis & Observations

> **[Student: write your personal reflections in this cell]**
>
> Suggested discussion points for AA4.3 and AA4.2:
> - What throughput did you observe? Is it consistent across batches? Why or why not?
> - Which step dominates latency — Bronze write, Silver MERGE, or Gold recompute? How did you determine this?
> - What is the trigger interval set to, and how does it affect throughput vs. latency trade-offs?
> - What would you change if this pipeline needed to handle 10× the data volume?
> - What are the limits of the `foreachBatch` approach compared to a native Delta streaming sink?
> - How does Kafka's `maxOffsetsPerTrigger` setting interact with batch size? What value did you choose and why?